# Pittsburgh street-closure category analysis
Reproducible experiment with six balanced categories, fixed prompt-development and held-out splits, and exact 0/1/several-example prompt conditions.

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version as package_version
import base64
import os
import json
import re
import subprocess
import sys

REPOSITORY_URL = "https://github.com/16howarthm/pgh-road-ai.git"
COLAB_REPOSITORY_DIR = Path("/content/pgh-road-ai")

def get_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def clone_private_repository(target):
    token = get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "GITHUB_TOKEN is unavailable. Add it to Colab Secrets, grant notebook access, "
            "and rerun this cell."
        )
    authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    clone_environment = os.environ.copy()
    clone_environment.pop("GITHUB_TOKEN", None)
    clone_environment.pop("OPENROUTER_API_KEY", None)
    clone_environment.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {authorization}",
        "GIT_TERMINAL_PROMPT": "0",
    })
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(target)],
            check=True, env=clone_environment,
        )
    finally:
        clone_environment["GIT_CONFIG_VALUE_0"] = ""
        authorization = None
        token = None

def is_project_root(path):
    required = ["road_ai.py", "pgh data.csv"]
    return all((path / name).exists() for name in required)

if is_project_root(Path.cwd()):
    PROJECT_ROOT = Path.cwd().resolve()
elif is_project_root(COLAB_REPOSITORY_DIR):
    PROJECT_ROOT = COLAB_REPOSITORY_DIR
elif Path("/content").exists():
    clone_private_repository(COLAB_REPOSITORY_DIR)
    PROJECT_ROOT = COLAB_REPOSITORY_DIR
else:
    raise FileNotFoundError("Run this notebook from the pgh-road-ai repository root.")

if PROJECT_ROOT == COLAB_REPOSITORY_DIR and (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "remote", "set-url", "origin", REPOSITORY_URL],
        check=True,
    )

os.chdir(PROJECT_ROOT)
required_packages = [
    ("pandas", "2.0"),
    ("scikit-learn", "1.3"),
    ("openai", "1.0"),
]
to_install = []
for package, minimum in required_packages:
    try:
        installed = package_version(package)
        installed_parts = tuple(map(int, re.findall(r"\d+", installed)[:2]))
        minimum_parts = tuple(map(int, minimum.split(".")))
        if installed_parts < minimum_parts:
            to_install.append(f"{package}>={minimum}")
    except PackageNotFoundError:
        to_install.append(f"{package}>={minimum}")
if to_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *to_install],
        check=True,
    )

import pandas as pd
from road_ai import *

if not os.getenv("OPENROUTER_API_KEY"):
    openrouter_key = get_secret("OPENROUTER_API_KEY")
    if openrouter_key:
        os.environ["OPENROUTER_API_KEY"] = openrouter_key
    openrouter_key = None

DATA_PATH = PROJECT_ROOT / "pgh data.csv"
SPLIT_DIR = PROJECT_ROOT / "category_splits"
OUTPUT_DIR = PROJECT_ROOT / "category_results"
LABEL_COL = "work_type"
CATEGORIES = DEFAULT_CATEGORIES
MODELS = ["meta-llama/llama-4-scout", "openai/gpt-oss-120b"]
PROMPT_CONDITIONS = {"zero": 0, "one": 1, "several": 6}
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("OpenRouter key available:", bool(os.getenv("OPENROUTER_API_KEY")))

## Load or create the fixed splits
The first run de-duplicates permits, reports and removes conflicting-label description groups, samples 50 unique descriptions per category, and saves fixed 120/180 splits. Later runs load those same records.

In [ ]:
prompt_path = SPLIT_DIR / "prompt_development.csv"
test_path = SPLIT_DIR / "heldout_test.csv"
audit_path = SPLIT_DIR / "split_audit.json"

if prompt_path.exists() and test_path.exists():
    prompt_dev, test = load_fixed_splits(LABEL_COL, SPLIT_DIR)
    audit = json.loads(audit_path.read_text()) if audit_path.exists() else {}
else:
    df = pd.read_csv(DATA_PATH)
    sample, audit = balanced_sample(
        df, LABEL_COL, CATEGORIES, n_per_category=50,
        random_state=RANDOM_STATE, return_audit=True,
    )
    prompt_dev, test = split_prompt_test(
        sample, LABEL_COL, prompt_n=120, test_n=180, random_state=RANDOM_STATE
    )
    save_fixed_splits(prompt_dev, test, LABEL_COL, SPLIT_DIR, audit)

validate_splits(prompt_dev, test, LABEL_COL)
print(audit)
print("Prompt-development counts:\n", prompt_dev[LABEL_COL].value_counts().sort_index())
print("Held-out counts:\n", test[LABEL_COL].value_counts().sort_index())

## Run every model and prompt condition
Each run uses the same saved 180-record held-out set. `one` contains exactly one labeled example total; `several` contains six total (one per category). Before running this cell in Colab, add `OPENROUTER_API_KEY` under the notebook's **Secrets** panel and grant notebook access.

In [ ]:
if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError(
        "OPENROUTER_API_KEY is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun the setup cell."
    )

all_results = []
for model in MODELS:
    for condition, n_examples in PROMPT_CONDITIONS.items():
        examples = make_examples(
            prompt_dev, LABEL_COL, n_examples=n_examples, random_state=RANDOM_STATE
        )
        assert len(examples) == n_examples
        run = run_experiment(
            test, LABEL_COL, model=model, categories=CATEGORIES, examples=examples
        )
        assert len(run) == 180
        assert run.permit_id.tolist() == test.permit_id.tolist()
        run["model"] = model
        run["prompt_condition"] = condition
        run["n_examples"] = n_examples
        all_results.append(run)

results = pd.concat(all_results, ignore_index=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "classification_results.csv", index=False)
print("Raw results saved to:", OUTPUT_DIR / "classification_results.csv")
results.head()

## Metrics and confusion matrices
Failures and invalid responses remain in the 180-row denominator and appear in the `__FAILED__` confusion-matrix column.

In [ ]:
summary_rows = []
matrix_rows = []
for (model, condition), group in results.groupby(["model", "prompt_condition"], sort=False):
    scores = score_results(group, CATEGORIES)
    assert scores["n_scored"] == 180
    summary_rows.append({
        "model": model,
        "prompt_condition": condition,
        "n_examples": int(group.n_examples.iloc[0]),
        "accuracy": scores["accuracy"],
        "macro_f1": scores["macro_f1"],
        "failed_predictions": scores["n_failures"],
        "prompt_tokens": group.prompt_tokens.sum(),
        "completion_tokens": group.completion_tokens.sum(),
    })
    matrix = scores["confusion_matrix"]
    display(model, condition, matrix)
    for actual, row in matrix.iterrows():
        for predicted, count in row.items():
            matrix_rows.append({
                "model": model, "prompt_condition": condition,
                "actual": actual, "predicted": predicted, "count": int(count),
            })

summary = pd.DataFrame(summary_rows)
confusion_matrices = pd.DataFrame(matrix_rows)
summary.to_csv(OUTPUT_DIR / "classification_summary.csv", index=False)
confusion_matrices.to_csv(OUTPUT_DIR / "confusion_matrices.csv", index=False)
print("All outputs saved under:", OUTPUT_DIR)
display(summary)

## Error review

In [ ]:
errors = results[results.actual.ne(results.predicted)].sort_values(
    ["model", "prompt_condition", "confidence"], na_position="first"
)
display(errors[[
    "model", "prompt_condition", "permit_id", "description",
    "actual", "predicted", "confidence", "reason", "error",
]].head(50))